In [3]:
# 智普AI

# LangChain的提示词模板引擎（Prompt Template Engine）是其核心组件之一，主要用于构建和管理与大预言模型（LLM）交互的提示词（Prompt）。
# 它的作用是将用户的输入或动态数据结构化地嵌入到预定义的模板中，从而生成适合模型处理的提示词，提升模型输出的准确性和一致性。

import os
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import warnings

warnings.filterwarnings("ignore", message=".*HMAC key is 16 bytes long.*")

key = os.getenv('ZHIPU_API_KEY')
os.environ["ZHIPUAI_API_KEY"] = key

chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5,
)


In [5]:
from langchain.prompts import PromptTemplate

# 定义模板
template = '你是一个{role}，请用{style}风格回答问题：{question}'
# 创建模板对象
prompt_template = PromptTemplate.from_template(template)

# 第一种写法变量的填充
# filled_prompt = prompt_template.format(role='数学老师', style='通俗易懂', question='勾股定理是什么？')
# 第二种写法
filled_prompt = prompt_template.invoke({'role': '数学老师', 'style': '通俗易懂', 'question': '勾股定理是什么？'})

# 模型响应数据
ai_msg = chat.invoke(filled_prompt)
print(ai_msg.content)

同学们好！今天咱们来聊一个特别实用的数学小知识——勾股定理。别被名字吓到，其实它特别简单，就像搭积木一样好理解！


### **先看个生活场景**  
想象一下，你站在教室的墙角，地面是正方形的，两面墙和地面形成了三个“边”。如果你从墙角出发，沿着地面走3米，再沿着墙面走4米，然后想直接从终点走回起点（也就是“斜着”穿过墙角），得走多远呢？  

这时候，勾股定理就派上用场啦！它告诉我们：**在直角三角形里（就是你墙角这种“方方正正”的三角形），两条直角边的平方和，等于斜边的平方**。  


### **“平方”是啥？别慌！**  
先解释“平方”：一个数的平方，就是它自己乘自己。比如3的平方是3×3=9，4的平方是4×4=16，5的平方是5×5=25。  


### **回到墙角的例子**  
你沿着地面走了3米（这是“直角边”之一），沿着墙面走了4米（这是“另一条直角边”），那斜着走的距离（“斜边”）怎么算呢？  

根据勾股定理：  
（直角边1）² + （直角边2）² = 斜边²  
也就是：3² + 4² = 斜边²  
算一下：9 + 16 = 25，所以斜边²=25，那斜边就是√25=5米！  
（√是“根号”，就是问“哪个数自己乘自己等于25”，答案就是5）  

你看，斜着走正好5米！是不是很神奇？  


### **再记个“口诀”**  
为了方便记忆，咱们编个口诀：  
**“勾三股四弦五”**  
这里的“勾”和“股”就是两条直角边，“弦”是斜边（古代数学家这么叫的，现在我们也用）。意思是：如果一条直角边是3，另一条是4，斜边一定是5。  

当然，不只是3、4、5，比如5和12：5²+12²=25+144=169，斜边就是√169=13；6和8：6²+8²=36+64=100，斜边就是10！只要是直角三角形，这个规律都成立。  


### **有啥用呢？**  
用处可大啦！  
- 量房子：要知道屋顶的斜边有多长，不用爬上去量，用勾股定理算就行；  
- 走路：从A点到B点，走“直角路线”是5+12=17米，走“斜线”只要13米，能省4米呢；  
- 甚至导航、盖房子、做设计，都离不开它！  


### **总结一下**  
勾股定理就是直角三角形的“身份证公式”：  
**两条直角边的平方和 = 斜边的平方**  
记作：a

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate

# 与上面的区别就是这种模板是以聊天的形式来交互

# 定义模板
sys_template = '你是数学老师，请以{style}的风格回答问题。'
user_template = '请用简单易懂的方式结束：{question}'

# 创建对话模板
prompt_template = ChatPromptTemplate.from_messages([
    ('system', sys_template),
    ('human', user_template)
])

# 填充变量
# prompt = prompt_template.format_messages(style='生动有趣', question='勾股定理是什么？')
prompt = prompt_template.invoke({'style':'生动有趣', 'question':'勾股定理是什么？'})

# 获取响应
msg = chat.invoke(prompt)
print(msg.content)

NameError: name 'chat' is not defined

### 输出格式化

In [4]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# 定义输出结构
response_schemas = [
    ResponseSchema(name="name", description="人的姓名", type="string"),
    ResponseSchema(name="age", description="人的年龄", type="integer"),
]

# 定义输出对象
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# 创建提示模板，包含格式化指令
template = """你是一个信息提取助手，请从以下文本中提取姓名和年龄，并以 JSON 格式返回：
文本：{input_text}
{format_instructions}
"""

prompt = PromptTemplate(
    template = template,
    partial_variables = {"format_instructions": output_parser.get_format_instructions()}
)

# 填充输入
input_text = "张三今年25岁，来自北京。"
filled_prompt = prompt.format(input_text = input_text)

# 获取输出
resp = chat.invoke(filled_prompt)

parsed_output = output_parser.parse(resp.content)
print(parsed_output)

{'name': '张三', 'age': 25}


In [7]:
# 转换成对象
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 定义输出模板
class Person(BaseModel):
    name: str = Field(description="人的姓名")
    age: int = Field(description="人的年龄")
    
# 创建输出解释器
output_parser = PydanticOutputParser(pydantic_object=Person)

# 创建提示模板
template = """你是一个信息提取助手，请从以下文本中提取姓名和年龄，并以 JSON 格式返回：
文本：{input_text}
{format_instructions}
"""

prompt = PromptTemplate(
    template = template,
    partial_variables = {"format_instructions": output_parser.get_format_instructions()}
)

# 填充输入
input_text = "张三今年25岁，来自北京。"
filled_prompt = prompt.format(input_text = input_text)

# 获取输出
resp = chat.invoke(filled_prompt)
parsed_output = output_parser.parse(resp.content)
parsed_output


Person(name='张三', age=25)

### LLMChain的核心组件

PromptTemplate:
- 定义提示词的模板，包含占位符（如{input}），用于动态生成提示词。
- 示例：“请使用{style}风格解释：{question}”

LLM:
- 大语言模型（如ChatZhipuAI、OpenAI等），负责根据提示词生成输出。
- 示例：ChatZhipuAI(model="glm-4")

OutputParser(可选)：
- 用于将LLM的原始文本输出解析为结构化格式（如JSON、列表）。
- 示例：StructuredOutputParser或PydanticOutputParser。

Memory(可选)：
- 可集成上下文记忆（如ConversationBufferMemory），支持对话历史管理。

### 链式调用

In [10]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

response_schemas = [
    ResponseSchema(name="answer", description="问题的答案", type="string"),
    ResponseSchema(name="confidence", description="答案的置信度", type="float")
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# 定义提示模板
template = """你是一名数学老师，请用{style}风格回答以下问题，并以 JSON 格式返回答案和置信度：
问题：{question}
{format_instructions}"""

prompt = PromptTemplate(
    template = template,
    input_variables=["style", "question"], # 可写可不写，会自动识别到style和question
    partial_variables={"format_instructions": output_parser.get_format_instructions()}
)

# 创建LLMChain（下面功能已经废弃了）
# llm_chain = LLMChain(llm=chat, prompt=prompt, output_parser=output_parser)
llm_chain = prompt | chat | output_parser # 推荐写法

# 运行链（下面功能已经废弃了）
# rs = llm_chain.run({
#     'style': '通俗易懂',
#     'question': '勾股定理是什么？'
# })

rs = llm_chain.invoke({
    'style': '通俗易懂',
    'question': '勾股定理是什么？'
})

print(rs)

{'answer': '勾股定理是一个关于直角三角形三边关系的数学定理。它说：在一个直角三角形中，两条直角边的长度的平方和，等于斜边长度的平方。用公式表示就是：a² + b² = c²，其中a和b是直角边，c是斜边。比如，一个直角三角形的两条直角边分别是3和4，那么斜边的长度就是5，因为3² + 4² = 9 + 16 = 25，而25是5的平方。', 'confidence': 1.0}


### 流式输出

In [13]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate

response_schemas = [
    ResponseSchema(name="answer", description="问题的答案", type="string"),
    ResponseSchema(name="confidence", description="答案的置信度", type="float")
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# 定义提示模板
template = """你是一名数学老师，请用{style}风格回答以下问题，并以 JSON 格式返回答案和置信度：
问题：{question}
{format_instructions}"""

prompt = PromptTemplate(
    template = template,
    partial_variables={"format_instructions": output_parser.get_format_instructions()}
)

# 创建LLMChain
llm_chain = prompt | chat # 不能加这个，否则流式输出还是会一把输出，因为他需要拿到结果才能格式化 | output_parser

chunks = [] # 存储输出
for chunk in llm_chain.stream({'style': '通俗易懂','question': '勾股定理是什么？'}):
    chunks.append(chunk)
    print(chunk.content, end='', flush=True)


```json
{
	"answer": "勾股定理是一个基本的几何定理，描述了直角三角形三条边之间的关系。具体来说，在一个直角三角形中，两条直角边的平方和等于斜边的平方。用公式表示就是：a² + b² = c²，其中a和b是直角边的长度，c是斜边的长度。这个定理在解决直角三角形的问题时非常有用，比如计算边长或判断一个三角形是否为直角三角形。",
	"confidence": 1.0
}
```